In [ ]:
# Core libs for file handling, arrays, dataframes, and plotting
import os  # filesystem paths and directory utilities
import numpy as np  # vectorized numerical ops
import pandas as pd  # tabular data handling
import seaborn as sns  # statistical plotting
sns.set_style('darkgrid')  # consistent dark grid background for plots
import matplotlib.pyplot as plt  # plotting primitives
from sklearn.model_selection import train_test_split  # dataset splitting helper
from sklearn.metrics import confusion_matrix, classification_report  # evaluation metrics

from PIL import Image, ImageEnhance, ImageFilter  # image I/O and basic manipulation

# Deep learning stack (TensorFlow/Keras) and computer vision helpers
import tensorflow as tf  # core TensorFlow engine
from tensorflow import keras  # Keras high-level API
from tensorflow.keras.models import Sequential  # linear stack of layers
from tensorflow.keras.optimizers import Adam, Adamax  # optimizers
from tensorflow.keras.metrics import categorical_crossentropy  # loss metric alias
from tensorflow.keras.preprocessing.image import ImageDataGenerator  # augmentation pipeline
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Activation, Dropout, BatchNormalization  # layer building blocks
from tensorflow.keras import regularizers  # weight regularization
import cv2  # OpenCV for image processing
import warnings  # control warnings
from tqdm import tqdm  # progress bars

from skimage.measure import shannon_entropy

from collections import defaultdict

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

%matplotlib inline
plt.rcParams.update({
    "figure.facecolor":  "#0D1117",
    "axes.facecolor":    "#0D1117",
    "axes.edgecolor":    "#21262D",
    "axes.labelcolor":   "#E6EDF3",
    "xtick.color":       "#E6EDF3",
    "ytick.color":       "#E6EDF3",
    "text.color":        "#E6EDF3",
    "grid.color":        "#21262D",
    "grid.alpha":        0.4,
    "font.family":       "monospace",
})

# Silence noisy warnings to keep notebook output clean
warnings.filterwarnings("ignore")

print('modules loaded')  # quick sanity check that imports succeeded

In [ ]:
DATASET1_ROOT   = "dataset/colon_dataset"   # → train/val/test → colon_aca, colon_n
DATASET2_ROOT   = "dataset"                 # → train/val/test → 0_normal, 1_ulcerative_colitis, 2_polyps
 
DATASET1_CLASSES = ["colon_aca", "colon_n"]
DATASET2_CLASSES = ["0_normal", "1_ulcerative_colitis", "2_polyps"]
SPLITS           = ["train", "val", "test"]
 
DS1_LABELS = {"colon_aca": "Adenocarcinoma", "colon_n": "Normal"}
DS2_LABELS = {
    "0_normal":              "Normal",
    "1_ulcerative_colitis":  "Ulcerative Colitis",
    "2_polyps":              "Polyps"
}

In [ ]:
CANCER_COLORS = ["#E63946", "#457B9D"]
GI_COLORS     = ["#2D6A4F", "#E9C46A", "#E76F51"]
SPLIT_COLORS  = ["#264653", "#2A9D8F", "#E9C46A"]
BG_COLOR      = "#0D1117"
TEXT_COLOR    = "#E6EDF3"
 
print("✅ Config set")

In [ ]:
def get_image_paths(root, classes, splits):
    data = defaultdict(lambda: defaultdict(list))
    for split in splits:
        for cls in classes:
            folder = os.path.join(root, split, cls)
            if os.path.isdir(folder):
                paths = [
                    os.path.join(folder, f)
                    for f in os.listdir(folder)
                    if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tiff"))
                ]
                data[split][cls] = paths
    return data
 
def count_images(data):
    return {s: {c: len(p) for c, p in cd.items()} for s, cd in data.items()}
 
def load_image_rgb(path, size=(224, 224)):
    return np.array(Image.open(path).convert("RGB").resize(size))
 
def sample_paths(data, cls, n=9):
    all_paths = []
    for split in data:
        all_paths.extend(data[split].get(cls, []))
    np.random.seed(42)
    return np.random.choice(all_paths, min(n, len(all_paths)), replace=False)
 
def collect_image_props(data, classes, n_per_class=100):
    records = []
    for cls in classes:
        all_paths = []
        for split in data:
            all_paths.extend(data[split].get(cls, []))
        np.random.seed(0)
        sampled = np.random.choice(all_paths, min(n_per_class, len(all_paths)), replace=False)
        for p in sampled:
            try:
                img = Image.open(p)
                w, h = img.size
                sz = os.path.getsize(p) / 1024
                records.append((w, h, sz, cls))
            except:
                pass
    return records
 
def props_arrays(props, classes):
    d = {c: {"w": [], "h": [], "sz": []} for c in classes}
    for w, h, sz, cls in props:
        d[cls]["w"].append(w)
        d[cls]["h"].append(h)
        d[cls]["sz"].append(sz)
    return d
 
def collect_pixels(data, classes, n=80, size=(128, 128)):
    result = {}
    for cls in classes:
        all_paths = []
        for split in data:
            all_paths.extend(data[split].get(cls, []))
        np.random.seed(7)
        sampled = np.random.choice(all_paths, min(n, len(all_paths)), replace=False)
        imgs = []
        for p in sampled:
            try:
                imgs.append(load_image_rgb(p, size))
            except:
                pass
        result[cls] = np.array(imgs) if imgs else np.zeros((1, *size, 3), dtype=np.uint8)
    return result
 
print("✅ Helper functions defined")

In [ ]:
ds1 = get_image_paths(DATASET1_ROOT, DATASET1_CLASSES, SPLITS)
ds2 = get_image_paths(DATASET2_ROOT, DATASET2_CLASSES, SPLITS)
counts1 = count_images(ds1)
counts2 = count_images(ds2)
 
print("📁 Dataset 1 — Colon Dataset")
for split in SPLITS:
    for cls, count in counts1[split].items():
        print(f"   {split:6s} | {DS1_LABELS[cls]:20s} → {count:,} images")
 
print("\n📁 Dataset 2 — GI Dataset")
for split in SPLITS:
    for cls, count in counts2[split].items():
        print(f"   {split:6s} | {DS2_LABELS[cls]:25s} → {count:,} images")

In [ ]:
# CELL 5 — VIZ 1: Class Distribution — Colon Dataset
# ─────────────────────────────────────────────────────────────
totals1 = {c: sum(counts1[s].get(c, 0) for s in SPLITS) for c in DATASET1_CLASSES}
 
fig, ax = plt.subplots(figsize=(8, 5))
fig.suptitle("VIZ 1 — Class Distribution: Colon Dataset",
             fontsize=13, fontweight="bold", color=TEXT_COLOR)
bars = ax.bar([DS1_LABELS[c] for c in DATASET1_CLASSES],
              list(totals1.values()), color=CANCER_COLORS, width=0.5, edgecolor=BG_COLOR)
for bar, val in zip(bars, totals1.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f"{val:,}", ha="center", fontsize=12, fontweight="bold", color=TEXT_COLOR)
ax.set_ylabel("Number of Images"); ax.set_xlabel("Class")
ax.yaxis.grid(True); ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

In [ ]:
# CELL 6 — VIZ 2: Class Distribution — GI Dataset
# ─────────────────────────────────────────────────────────────
totals2 = {c: sum(counts2[s].get(c, 0) for s in SPLITS) for c in DATASET2_CLASSES}
 
fig, ax = plt.subplots(figsize=(9, 5))
fig.suptitle("VIZ 2 — Class Distribution: GI Dataset",
             fontsize=13, fontweight="bold", color=TEXT_COLOR)
bars = ax.bar([DS2_LABELS[c] for c in DATASET2_CLASSES],
              list(totals2.values()), color=GI_COLORS, width=0.5, edgecolor=BG_COLOR)
for bar, val in zip(bars, totals2.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f"{val:,}", ha="center", fontsize=12, fontweight="bold", color=TEXT_COLOR)
ax.set_ylabel("Number of Images"); ax.set_xlabel("Class")
ax.yaxis.grid(True); ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

In [ ]:
# CELL 7 — VIZ 3: Train/Val/Test Split — Colon Dataset
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
fig.suptitle("VIZ 3 — Train / Val / Test Split: Colon Dataset",
             fontsize=13, fontweight="bold", color=TEXT_COLOR)
x = np.arange(len(DATASET1_CLASSES))
bottoms = np.zeros(len(DATASET1_CLASSES))
for split, color in zip(SPLITS, SPLIT_COLORS):
    vals = [counts1[split].get(c, 0) for c in DATASET1_CLASSES]
    ax.bar(x, vals, bottom=bottoms, color=color, label=split.capitalize(), edgecolor=BG_COLOR)
    for j, (v, b) in enumerate(zip(vals, bottoms)):
        if v > 0:
            ax.text(x[j], b + v/2, str(v), ha="center", va="center",
                    fontsize=9, fontweight="bold", color="white")
    bottoms += np.array(vals)
ax.set_xticks(x)
ax.set_xticklabels([DS1_LABELS[c] for c in DATASET1_CLASSES])
ax.set_ylabel("Image Count")
ax.legend(loc="upper right")
ax.yaxis.grid(True); ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

In [ ]:
# CELL 8 — VIZ 4: Train/Val/Test Split — GI Dataset
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle("VIZ 4 — Train / Val / Test Split: GI Dataset",
             fontsize=13, fontweight="bold", color=TEXT_COLOR)
x = np.arange(len(DATASET2_CLASSES))
bottoms = np.zeros(len(DATASET2_CLASSES))
for split, color in zip(SPLITS, SPLIT_COLORS):
    vals = [counts2[split].get(c, 0) for c in DATASET2_CLASSES]
    ax.bar(x, vals, bottom=bottoms, color=color, label=split.capitalize(), edgecolor=BG_COLOR)
    for j, (v, b) in enumerate(zip(vals, bottoms)):
        if v > 0:
            ax.text(x[j], b + v/2, str(v), ha="center", va="center",
                    fontsize=9, fontweight="bold", color="white")
    bottoms += np.array(vals)
ax.set_xticks(x)
ax.set_xticklabels([DS2_LABELS[c] for c in DATASET2_CLASSES])
ax.set_ylabel("Image Count")
ax.legend(loc="upper right")
ax.yaxis.grid(True); ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

In [ ]:
# CELL 9 — VIZ 5: Pie Chart — Overall Distribution Both Datasets
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
fig.suptitle("VIZ 5 — Overall Class Distribution (Both Datasets)",
             fontsize=13, fontweight="bold", color=TEXT_COLOR)
for ax, (totals, labels, colors, title) in zip(axes, [
    (totals1, [DS1_LABELS[c] for c in DATASET1_CLASSES], CANCER_COLORS, "Colon Dataset"),
    (totals2, [DS2_LABELS[c] for c in DATASET2_CLASSES], GI_COLORS,     "GI Dataset"),
]):
    wedges, texts, autotexts = ax.pie(
        list(totals.values()), labels=labels, colors=colors,
        autopct="%1.1f%%", startangle=140, pctdistance=0.75,
        wedgeprops=dict(edgecolor=BG_COLOR, linewidth=2))
    for t in texts + autotexts:
        t.set_color(TEXT_COLOR); t.set_fontsize(10)
    ax.set_title(title, fontsize=11, fontweight="bold", color=TEXT_COLOR)
plt.tight_layout()
plt.show()

In [ ]:
# CELL 10 — VIZ 6 & 7: Sample Images — Colon Dataset
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 9, figsize=(22, 6))
fig.suptitle("VIZ 6 & 7 — Sample Images: Colon Dataset\n"
             "Top: Adenocarcinoma (Cancer)   |   Bottom: Normal Tissue",
             fontsize=12, fontweight="bold", color=TEXT_COLOR)
 
for row, (cls, color) in enumerate(zip(DATASET1_CLASSES, CANCER_COLORS)):
    paths = sample_paths(ds1, cls, n=9)
    for ax, path in zip(axes[row], paths):
        try:
            ax.imshow(Image.open(path).convert("RGB"))
        except:
            ax.set_facecolor("#222")
        ax.axis("off")
        for spine in ax.spines.values():
            spine.set_edgecolor(color); spine.set_linewidth(2); spine.set_visible(True)
 
plt.tight_layout()
plt.show()

In [ ]:
# CELL 11 — VIZ 8, 9, 10: Sample Images — GI Dataset
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 9, figsize=(22, 9))
fig.suptitle("VIZ 8, 9, 10 — Sample Images: GI Dataset\n"
             "Row 1: Normal  |  Row 2: Ulcerative Colitis  |  Row 3: Polyps",
             fontsize=12, fontweight="bold", color=TEXT_COLOR)
 
for row, (cls, color) in enumerate(zip(DATASET2_CLASSES, GI_COLORS)):
    paths = sample_paths(ds2, cls, n=9)
    for ax, path in zip(axes[row], paths):
        try:
            ax.imshow(Image.open(path).convert("RGB"))
        except:
            ax.set_facecolor("#222")
        ax.axis("off")
        for spine in ax.spines.values():
            spine.set_edgecolor(color); spine.set_linewidth(2); spine.set_visible(True)
 
plt.tight_layout()
plt.show()

In [ ]:
#  CATEGORY 3 — IMAGE PROPERTIES
# ══════════════════════════════════════════════════════════════
 
# ─────────────────────────────────────────────────────────────
# CELL 12 — Collect image property data (run once)
# ─────────────────────────────────────────────────────────────
props1 = collect_image_props(ds1, DATASET1_CLASSES)
props2 = collect_image_props(ds2, DATASET2_CLASSES)
pa1 = props_arrays(props1, DATASET1_CLASSES)
pa2 = props_arrays(props2, DATASET2_CLASSES)
print(f"✅ Collected properties for {len(props1)} images (DS1) and {len(props2)} images (DS2)")

In [ ]:
# CELL 13 — VIZ 11 & 12: Resolution Scatter — Both Datasets
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("VIZ 11 & 12 — Image Resolution Distribution",
             fontsize=13, fontweight="bold", color=TEXT_COLOR)
 
for ax, (pa, classes, labels_map, colors, title) in zip(axes, [
    (pa1, DATASET1_CLASSES, DS1_LABELS, CANCER_COLORS, "Colon Dataset"),
    (pa2, DATASET2_CLASSES, DS2_LABELS, GI_COLORS,     "GI Dataset"),
]):
    for cls, color in zip(classes, colors):
        ax.scatter(pa[cls]["w"], pa[cls]["h"], alpha=0.6, color=color,
                   label=labels_map[cls], s=30, edgecolors="none")
    ax.set_xlabel("Width (px)"); ax.set_ylabel("Height (px)")
    ax.set_title(title, color=TEXT_COLOR); ax.legend(); ax.grid(True)
 
plt.tight_layout()
plt.show()

In [ ]:
# CELL 14 — VIZ 13 & 14: File Size Distribution — Both Datasets
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("VIZ 13 & 14 — File Size Distribution (KB)",
             fontsize=13, fontweight="bold", color=TEXT_COLOR)
 
for ax, (pa, classes, labels_map, colors, title) in zip(axes, [
    (pa1, DATASET1_CLASSES, DS1_LABELS, CANCER_COLORS, "Colon Dataset"),
    (pa2, DATASET2_CLASSES, DS2_LABELS, GI_COLORS,     "GI Dataset"),
]):
    for cls, color in zip(classes, colors):
        ax.hist(pa[cls]["sz"], bins=40, alpha=0.7, color=color,
                label=labels_map[cls], edgecolor="none")
    ax.set_xlabel("File Size (KB)"); ax.set_ylabel("Frequency")
    ax.set_title(title, color=TEXT_COLOR); ax.legend(); ax.grid(True)
 
plt.tight_layout()
plt.show()

In [ ]:
# CELL 15 — VIZ 15: Aspect Ratio Distribution — Both Datasets
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("VIZ 15 — Aspect Ratio Distribution (W/H)",
             fontsize=13, fontweight="bold", color=TEXT_COLOR)
 
for ax, (pa, classes, labels_map, colors, title) in zip(axes, [
    (pa1, DATASET1_CLASSES, DS1_LABELS, CANCER_COLORS, "Colon Dataset"),
    (pa2, DATASET2_CLASSES, DS2_LABELS, GI_COLORS,     "GI Dataset"),
]):
    for cls, color in zip(classes, colors):
        ratios = [w/h for w, h in zip(pa[cls]["w"], pa[cls]["h"]) if h > 0]
        ax.hist(ratios, bins=30, alpha=0.7, color=color,
                label=labels_map[cls], edgecolor="none")
    ax.set_xlabel("Aspect Ratio (W/H)"); ax.set_ylabel("Frequency")
    ax.set_title(title, color=TEXT_COLOR); ax.legend(); ax.grid(True)
 
plt.tight_layout()
plt.show()

In [ ]:
#  CATEGORY 4 — PIXEL & COLOR ANALYSIS
# ══════════════════════════════════════════════════════════════
 
# ─────────────────────────────────────────────────────────────
# CELL 16 — Collect pixel data (run once, takes ~30 seconds)
# ─────────────────────────────────────────────────────────────
print("⏳ Collecting pixel data, please wait …")
pixels1 = collect_pixels(ds1, DATASET1_CLASSES)
pixels2 = collect_pixels(ds2, DATASET2_CLASSES)
print("✅ Pixel data ready")

In [ ]:
# CELL 17 — VIZ 16 & 17: Mean Pixel Intensity — Both Datasets
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("VIZ 16 & 17 — Mean Pixel Intensity per Class",
             fontsize=13, fontweight="bold", color=TEXT_COLOR)
 
for ax, (pixels, classes, labels_map, colors, title) in zip(axes, [
    (pixels1, DATASET1_CLASSES, DS1_LABELS, CANCER_COLORS, "Colon Dataset"),
    (pixels2, DATASET2_CLASSES, DS2_LABELS, GI_COLORS,     "GI Dataset"),
]):
    means = [pixels[c].mean() for c in classes]
    stds  = [pixels[c].std()  for c in classes]
    bars = ax.bar([labels_map[c] for c in classes], means, yerr=stds,
                  color=colors, capsize=6, width=0.5, edgecolor=BG_COLOR)
    for bar, m in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                f"{m:.1f}", ha="center", fontsize=10, fontweight="bold", color=TEXT_COLOR)
    ax.set_ylabel("Mean Pixel Value (0–255)"); ax.set_ylim(0, 290)
    ax.set_title(title, color=TEXT_COLOR)
    ax.yaxis.grid(True); ax.set_axisbelow(True)
 
plt.tight_layout()
plt.show()

In [ ]:
# CELL 18 — VIZ 18 & 19: RGB Channel Histograms — Both Datasets
# ─────────────────────────────────────────────────────────────
channel_colors = ["#FF6B6B", "#6BCB77", "#4D96FF"]
 
fig, axes = plt.subplots(2, max(len(DATASET1_CLASSES), len(DATASET2_CLASSES)),
                         figsize=(16, 8))
fig.suptitle("VIZ 18 & 19 — RGB Channel Histograms\nTop: Colon Dataset  |  Bottom: GI Dataset",
             fontsize=12, fontweight="bold", color=TEXT_COLOR)
 
for row, (pixels, classes, labels_map) in enumerate([
    (pixels1, DATASET1_CLASSES, DS1_LABELS),
    (pixels2, DATASET2_CLASSES, DS2_LABELS),
]):
    for col, cls in enumerate(classes):
        ax = axes[row][col]
        imgs = pixels[cls]
        for ch, ch_color, ch_name in zip(range(3), channel_colors, ["R", "G", "B"]):
            vals = imgs[:, :, :, ch].flatten()
            counts_h, bin_edges = np.histogram(vals, bins=64, range=(0, 255))
            ax.plot(bin_edges[:-1], counts_h, color=ch_color,
                    label=ch_name, linewidth=1.5, alpha=0.9)
        ax.set_title(labels_map[cls], color=TEXT_COLOR, fontsize=9)
        ax.legend(fontsize=8); ax.grid(True)
        ax.set_xlabel("Pixel Value", fontsize=8)
        ax.set_ylabel("Frequency", fontsize=8)
    # hide unused axes in row 0 if DS1 has fewer classes
    for col in range(len(classes), axes.shape[1]):
        axes[row][col].set_visible(False)
 
plt.tight_layout()
plt.show()

In [ ]:
# CELL 19 — VIZ 20: Mean Image per Class — Both Datasets
# ─────────────────────────────────────────────────────────────
all_classes = DATASET1_CLASSES + DATASET2_CLASSES
all_pixels  = {**pixels1, **pixels2}
all_labels  = {**DS1_LABELS, **DS2_LABELS}
 
fig, axes = plt.subplots(1, len(all_classes), figsize=(18, 4))
fig.suptitle("VIZ 20 — Mean Image per Class (Both Datasets)",
             fontsize=13, fontweight="bold", color=TEXT_COLOR)
for ax, cls in zip(axes, all_classes):
    mean_img = all_pixels[cls].mean(axis=0).astype(np.uint8)
    ax.imshow(mean_img)
    ax.set_title(all_labels[cls], fontsize=9, fontweight="bold", color=TEXT_COLOR)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
#  CATEGORY 5 — ADVANCED ANALYSIS
# ══════════════════════════════════════════════════════════════
 
# ─────────────────────────────────────────────────────────────
# CELL 20 — VIZ 21: Per-Channel Intensity Heatmap
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("VIZ 21 — Per-Channel Mean Intensity Heatmap",
             fontsize=13, fontweight="bold", color=TEXT_COLOR)
 
for ax, (pixels, classes, labels_map, title) in zip(axes, [
    (pixels1, DATASET1_CLASSES, DS1_LABELS, "Colon Dataset"),
    (pixels2, DATASET2_CLASSES, DS2_LABELS, "GI Dataset"),
]):
    matrix = np.array([
        [pixels[c][:, :, :, ch].mean() for ch in range(3)]
        for c in classes
    ])
    sns.heatmap(matrix,
                xticklabels=["R", "G", "B"],
                yticklabels=[labels_map[c] for c in classes],
                annot=True, fmt=".1f", cmap="magma",
                linewidths=0.5, ax=ax, cbar_kws={"shrink": 0.8})
    ax.set_title(title, fontsize=11, color=TEXT_COLOR)
 
plt.tight_layout()
plt.show()

In [ ]:
# CELL 21 — Collect PCA pixel data (run once)
# ─────────────────────────────────────────────────────────────
print("⏳ Collecting PCA pixel data …")
SIZE_PCA = (64, 64)
pca_pix1 = collect_pixels(ds1, DATASET1_CLASSES, n=120, size=SIZE_PCA)
pca_pix2 = collect_pixels(ds2, DATASET2_CLASSES, n=120, size=SIZE_PCA)
 
def build_pca_data(pixels, classes):
    X, y = [], []
    for i, cls in enumerate(classes):
        imgs = pixels[cls]
        flat = imgs.reshape(len(imgs), -1).astype(np.float32) / 255.0
        X.append(flat); y.extend([i] * len(flat))
    X = np.vstack(X)
    X_scaled = StandardScaler().fit_transform(X)
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X_scaled)
    return X_pca, np.array(y), pca.explained_variance_ratio_
 
X_pca1, y1, evr1 = build_pca_data(pca_pix1, DATASET1_CLASSES)
X_pca2, y2, evr2 = build_pca_data(pca_pix2, DATASET2_CLASSES)
print("✅ PCA computed")

In [ ]:
# CELL 22 — VIZ 22 & 23: PCA Feature Space — Both Datasets
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("VIZ 22 & 23 — PCA Feature Space (Both Datasets)",
             fontsize=13, fontweight="bold", color=TEXT_COLOR)
 
for ax, (X_pca, y, evr, classes, labels_map, colors, title) in zip(axes, [
    (X_pca1, y1, evr1, DATASET1_CLASSES, DS1_LABELS, CANCER_COLORS, "Colon Dataset"),
    (X_pca2, y2, evr2, DATASET2_CLASSES, DS2_LABELS, GI_COLORS,     "GI Dataset"),
]):
    for i, (cls, color) in enumerate(zip(classes, colors)):
        mask = y == i
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1], color=color,
                   label=labels_map[cls], alpha=0.7, s=25, edgecolors="none")
    ax.set_xlabel(f"PC1 ({evr[0]*100:.1f}% variance)")
    ax.set_ylabel(f"PC2 ({evr[1]*100:.1f}% variance)")
    ax.set_title(title, color=TEXT_COLOR); ax.legend(); ax.grid(True)
 
plt.tight_layout()
plt.show()

In [ ]:
# CELL 23 — VIZ 24: Brightness & Contrast Distribution
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("VIZ 24 — Brightness & Contrast Distribution per Class",
             fontsize=13, fontweight="bold", color=TEXT_COLOR)
 
for row, (pixels, classes, labels_map, colors, title) in enumerate([
    (pixels1, DATASET1_CLASSES, DS1_LABELS, CANCER_COLORS, "Colon Dataset"),
    (pixels2, DATASET2_CLASSES, DS2_LABELS, GI_COLORS,     "GI Dataset"),
]):
    ax_b, ax_c = axes[row][0], axes[row][1]
    for cls, color in zip(classes, colors):
        imgs_gray  = pixels[cls].mean(axis=-1)
        brightness = imgs_gray.mean(axis=(1, 2))
        contrast   = imgs_gray.std(axis=(1, 2))
        ax_b.hist(brightness, bins=30, alpha=0.65, color=color,
                  label=labels_map[cls], edgecolor="none")
        ax_c.hist(contrast,   bins=30, alpha=0.65, color=color,
                  label=labels_map[cls], edgecolor="none")
    ax_b.set_title(f"Brightness — {title}", color=TEXT_COLOR)
    ax_b.set_xlabel("Mean Pixel Value"); ax_b.set_ylabel("Frequency")
    ax_b.legend(); ax_b.grid(True)
    ax_c.set_title(f"Contrast — {title}", color=TEXT_COLOR)
    ax_c.set_xlabel("Std Dev of Pixels"); ax_c.set_ylabel("Frequency")
    ax_c.legend(); ax_c.grid(True)
 
plt.tight_layout()
plt.show()

In [ ]:
# CELL 24 — VIZ 25: Augmentation Preview
# ─────────────────────────────────────────────────────────────
def augment_image(img_arr):
    img = Image.fromarray(img_arr)
    return {
        "Original":          img,
        "Horiz. Flip":       img.transpose(Image.FLIP_LEFT_RIGHT),
        "Vert. Flip":        img.transpose(Image.FLIP_TOP_BOTTOM),
        "Rotate 90°":        img.rotate(90),
        "Brightness +50%":   ImageEnhance.Brightness(img).enhance(1.5),
        "Contrast +50%":     ImageEnhance.Contrast(img).enhance(1.5),
        "Blur":              img.filter(ImageFilter.GaussianBlur(radius=2)),
        "Zoom (crop)":       img.crop((img.width//6, img.height//6,
                                       img.width*5//6, img.height*5//6)).resize(img.size),
    }
 
sample1_path = sample_paths(ds1, "colon_aca", n=1)[0]
sample2_path = sample_paths(ds2, "2_polyps",  n=1)[0]
 
fig, axes = plt.subplots(2, 8, figsize=(22, 6))
fig.suptitle("VIZ 25 — Data Augmentation Preview\n"
             "Top: Colon Adenocarcinoma  |  Bottom: Polyps",
             fontsize=12, fontweight="bold", color=TEXT_COLOR)
 
for row, path in enumerate([sample1_path, sample2_path]):
    img_arr = load_image_rgb(path, (224, 224))
    augs = augment_image(img_arr)
    for ax, (name, aug_img) in zip(axes[row], augs.items()):
        ax.imshow(aug_img)
        ax.set_title(name, fontsize=8, fontweight="bold", color=TEXT_COLOR)
        ax.axis("off")
 
plt.tight_layout()
plt.show()